In [1]:
import os

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path 

In [3]:
%pwd

'd:\\data_pipeline\\notebooks'

In [4]:
os.chdir("../")

#data class ko config.yaml se read karna hai


In [5]:
from src.data_line.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH, SCHEMA_FILE_PATH
from src.data_line.utils.common import read_yaml, create_directories

In [6]:
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH,
                 schema_filepath=SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])
        
        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config.root_dir),
            source_URL=config.source_URL,
            local_data_file=Path(config.local_data_file),
            unzip_dir=Path(config.unzip_dir)
        )

        return data_ingestion_config    

In [7]:
import os
import urllib.request as request
from src.data_line import logger
import zipfile

In [8]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_data(self):
        if not os.path.exists(self.config.local_data_file):
           filename, headers=request.urlretrieve(url=self.config.source_URL, filename=self.config.local_data_file)
           logger.info(f"File downloaded successfully: {filename}")
        else:
            logger.info(f"File already exists at location: {self.config.local_data_file}, skipping download.")


    def extract_zip_file(self):
        """zip_file_path: str
           Extracts the zip file into the data directory
           Function return None
        """
        unzip_path=self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

            
        

In [12]:
try:  
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_data()
    #data_ingestion.extract_zip_file()
except Exception as e:
    raise e

2026-02-10 01:32:12,178 - INFO - common - yaml file: config\config.yaml loaded successfully
2026-02-10 01:32:12,189 - INFO - common - yaml file: params.yaml loaded successfully
2026-02-10 01:32:12,197 - INFO - common - yaml file: schema.yaml loaded successfully
2026-02-10 01:32:12,203 - INFO - common - created directory at: artifacts
2026-02-10 01:32:12,208 - INFO - common - created directory at: artifacts/data_ingestion
2026-02-10 01:32:12,211 - INFO - 3889372301 - File already exists at location: artifacts\data_ingestion\data.zip, skipping download.
